# 04 — Complete SAGE-AVO model and training

| Item | Definition |
|---|---|
| **Scientific purpose** | Train a structure-aware graph/CNN model to refine a supplied low-frequency elastic prior using near/mid/far AVO. |
| **Inputs** | Notebook-03 patch index, train-only normalization, realization splits, AVO, low prior, RGT, elastic targets, segmentation targets, and masks. |
| **Outputs** | Criterion-specific fixed-objective, sampled, segmentation, whole-realization, periodic, and resumable final checkpoints with complete raw/weighted metric logs. |
| **Data availability** | Architecture, losses, and orchestration are public; datasets and checkpoints remain private/local. |
| **Local data requirements** | Completed Stage-03 artifacts and adequate PyTorch/PyG compute. Execution stops if the dataset contract is unavailable. |
| **Software requirements** | `pip install -e ".[ml,notebooks]"` with PyTorch and PyTorch Geometric. |
| **Approximate runtime** | Operator checks: seconds to minutes. Production 120-epoch SAGE-AVO training is a GPU-scale job. |
| **Pipeline position** | Consumes Notebook 03; produces matched checkpoints and manifests evaluated in Notebook 05. |

The implemented transport is a deterministic straight-path conditional residual flow from the low-frequency prior toward the target. It is **not** a calibrated probabilistic posterior. The graph module is PyTorch Geometric `TransformerConv` graph attention/message passing—not a full-image Vision Transformer.

In [ ]:
from pathlib import Path

def find_repository_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "sage_avo").exists():
            return candidate
    raise RuntimeError("SAGE-AVO repository root not found; start the kernel within an installed checkout.")

ROOT = find_repository_root()

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

from sage_avo.config import load_config, seed_everything
from sage_avo.data import IndexedRealizationPatches
from sage_avo.models import build_sage_avo_variant, sage_avo_model_kwargs
from sage_avo.models.sage_avo import angular_features
from sage_avo.training.engine import PhysicsNormalization, train_step
from sage_avo.training.flow import straight_path
from sage_avo.experiments.training import (
    curriculum_from_config,
    loss_weights_from_config,
    physics_settings_from_config,
    train_controlled_variant,
)

workflow_path = ROOT / "configs" / "sage_avo_s01_v003.yaml"
paths_file = ROOT / "configs" / "paths.yaml"
if not paths_file.exists():
    raise FileNotFoundError("Missing local configuration: configs/paths.yaml (template: configs/paths.example.yaml).")
paths = load_config(paths_file)
private_root = Path(paths["private_artifact_root"])
validation_root_text = os.getenv("SAGE_AVO_REVISION3_VALIDATION_ROOT", "").strip()
if validation_root_text:
    validation_root = Path(validation_root_text)
    workflow_path = validation_root / "configs" / "training_resolved.json"
    workflow = json.loads(workflow_path.read_text())
    dataset_dir = validation_root / "stage03" / "dataset"
    experiment_dir = validation_root / "stage04" / "sage_avo_s01_v003_stage01v003_validation8"
    figure_dir = validation_root / "figures" / "stage04"
else:
    workflow = load_config(workflow_path)
    dataset_dir = private_root / "stage_artifacts" / "stage03" / "ds_v003_production100_diverse" / "dataset"
    experiment_dir = private_root / "stage_artifacts" / "stage04" / "sage_avo_s01_v003_production"
    figure_dir = private_root / "figures" / "revision3" / "stage04"
seed_everything(int(workflow["experiment"]["seed"]))
figure_dir.mkdir(parents=True, exist_ok=True)

## 1. Immutable data contract

Each training item contains normalized low/mid/high AVO `[3,H,W]`, normalized low-frequency Vp/Vs/density `[3,H,W]`, normalized elastic target `[3,H,W]`, RGT `[H,W]`, segmentation `[H,W]`, valid mask `[1,H,W]`, and traceability metadata. Normalization is fitted only on training realizations and its elastic statistics are also installed in the model for physical-unit forward modeling and optional sampling guidance.

In [ ]:
if not (dataset_dir / "dataset_manifest.json").exists():
    raise FileNotFoundError(f"Stage-03 dataset manifest not found: {dataset_dir / 'dataset_manifest.json'}")
dataset_manifest = json.loads((dataset_dir / "dataset_manifest.json").read_text())
normalization = json.loads((dataset_dir / "normalization.json").read_text())
split_ids = json.loads((dataset_dir / "split_ids.json").read_text())
train_data = IndexedRealizationPatches(dataset_dir, "train")
validation_data = IndexedRealizationPatches(dataset_dir, "validation")
sample = train_data[0]
display(pd.Series({
    "train_patches": len(train_data),
    "validation_patches": len(validation_data),
    "split_unit": dataset_manifest["split_unit"],
    "prior_truth_derived": dataset_manifest["prior"]["truth_derived"],
}).to_frame("value"))
print({key: tuple(value.shape) for key, value in sample.items() if isinstance(value, torch.Tensor)})

## 2. Compact AVO summaries

The three stacks are retained as image channels. A least-squares line in `sin²(theta)` additionally yields intercept `P` and gradient `G`; near–2×mid+far supplies curvature. These summaries condition graph edges and node features. They are feature extraction—not the Stage-02 forward model.

In [ ]:
representative_angles = tuple(float(value) for value in workflow["model"]["representative_angles_degrees"])
with torch.no_grad():
    angular, gradient = angular_features(sample["avo"].unsqueeze(0), representative_angles)
print("[near, mid, far, P, G, curvature] shape:", tuple(angular.shape))

fig, axes = plt.subplots(1, 5, figsize=(15, 3), constrained_layout=True)
for axis, panel, title in zip(
    axes,
    [sample["avo"][0], sample["avo"][1], sample["avo"][2], angular[0, 3], angular[0, 4]],
    ["Near", "Mid", "Far", "Intercept P", "Gradient G"],
):
    axis.imshow(panel, aspect="auto", cmap="coolwarm")
    axis.set_title(title); axis.set_xticks([]); axis.set_yticks([])
feature_path = figure_dir / "stage04_avo_feature_contract.png"
fig.savefig(feature_path, dpi=300, bbox_inches="tight")
plt.show()

## 3. CNN, RGT-steered graph, and reinjection

1. A CNN encodes the current elastic state, time, three AVO bands, and low-frequency prior.
2. Every image sample is a graph node.
3. Edges include vertical trace neighbors and bidirectional lateral neighbors. For an RGT-steered edge, the adjacent-trace endpoint is chosen within ±3 time samples by minimum RGT mismatch; the no-RGT control uses Cartesian lateral neighbors.
4. Edge attributes decrease with local AVO-gradient contrast.
5. Two `TransformerConv` layers perform graph attention/message passing and return learned final-layer attention.
6. Graph features are reshaped to the image grid, reinjected into CNN features, and decoded into elastic transport velocity; a second decoder predicts shale/sand/plume classes.

The canonical architecture uses a two-block CNN, an RGT-steered dynamic graph, two four-head `TransformerConv` layers, graph-to-image residual reinjection, an elastic velocity decoder, and a convolutional segmentation decoder.

In [ ]:
model_config = workflow["model"]
full = build_sage_avo_variant("full", **sage_avo_model_kwargs(workflow)).eval()
full.set_norm_stats(normalization)
display(pd.Series({
    "graph_mode": full.graph_mode,
    "trainable_parameters": sum(parameter.numel() for parameter in full.parameters()),
    "graph_layers": model_config["graph_layers"],
    "graph_heads": model_config["graph_heads"],
}).to_frame("value"))
with torch.no_grad():
    state = sample["low"].unsqueeze(0)
    output = full(state, torch.zeros(1), sample["avo"].unsqueeze(0), state, sample["rgt"].unsqueeze(0))
print("elastic velocity:", tuple(output.velocity.shape))
print("segmentation logits:", tuple(output.segmentation_logits.shape))
print("graph embedding:", tuple(output.embeddings.shape))
print("directed graph edges:", output.edge_indices[0].shape[1])
print("edge attributes:", tuple(output.edge_weights[0].shape))
print("learned attention:", tuple(output.attention_weights[0].shape))

## 4. Deterministic conditional residual transport

Training samples a time `t ~ Uniform(0,1)` and constructs

\[
x_t=(1-t)x_{low}+t y,\qquad u^*=y-x_{low}.
\]

The network predicts the straight-path velocity conditioned on AVO, the supplied prior, and RGT. Inference starts at `x_low` and integrates the learned velocity from `t=0` to `1` with Heun/RK2 steps. Optional physics guidance differentiates exact-PP AVO mismatch with respect to the current elastic state and corrects selected trajectory steps. The production configuration sets `guidance_scale=0.0`. No stochastic base distribution or posterior calibration is implemented.

In [ ]:
t = torch.tensor([0.35])
state, target_velocity = straight_path(
    sample["low"].unsqueeze(0), sample["target"].unsqueeze(0), t
)
assert torch.allclose(target_velocity, sample["target"].unsqueeze(0) - sample["low"].unsqueeze(0))
print("state and velocity:", tuple(state.shape), tuple(target_velocity.shape))

## 5. Complete training objective

\[
L = w_{inv}(0.65L_{flow}+0.20L_{property}+w_{ssim}L_{SSIM})
    +0.30L_{seg}+w_cL_{contrastive}+w_{phys}L_{Zoeppritz}+w_gL_{graph}.
\]

`L_flow` fits residual velocity and uses a density weight increasing from 2.0 to 3.5. `L_property` supervises the reconstructed endpoint `x_low + velocity`. Masked SSIM decreases from 0.15 to 0.05. Segmentation combines masked class-weighted cross-entropy and masked Dice. `L_Zoeppritz` compares the native central crop with stored **clean** Stage-02 bands while using truth elastic halo, global sample origin, and the identical hashed exact-PP/wavelet/mute contract. Complex post-critical slowness follows the Stage-02 convention. Multiscale patches retain supervised losses but receive zero exact-physics mask. `L_graph` penalizes elastic contrast along high-weight edges. Physics and structural weights decay to 70% and 75% of their initial values. Contrastive and adaptive task weighting are implemented capabilities but are disabled in the v003 configuration.

Stage 02 and Stage 04 use the same declared shared-endpoint bands (`3–17`, `17–31`, `31–45`). The overlap at 17° and 31° is intentional. Compact P/G representative angles are the corresponding band midpoints (`10°`, `24°`, `38°`).

In [ ]:
training = workflow["training"]
display(pd.Series(training["loss_weights"], name="weight").to_frame())
display(pd.DataFrame(training["curriculum"]).T)
display(pd.DataFrame.from_dict(workflow["capabilities"], orient="index"))
display(pd.Series({
    "optimizer": "AdamW",
    "learning_rate": training["learning_rate"],
    "weight_decay": training["weight_decay"],
    "scheduler": "cosine annealing",
    "epochs": training["epochs"],
    "checkpoint_criteria": training["checkpoint_criteria"],
}).to_frame("value"))

## 6. Real-batch operator validation

This cell performs one optimization step on a real Stage-03 batch using the production model and losses. It checks gradients, exact differentiable forward consistency, graph construction, and tensor contracts; it is not reported as a trained result.

In [ ]:
native_index = int(train_data.index.index[train_data.index["physics_eligible"] == 1][0])
loader = DataLoader(Subset(train_data, [native_index]), batch_size=1, shuffle=False, num_workers=0)
batch = next(iter(loader))
assert bool(batch["physics_eligible"].all())
operator_model = build_sage_avo_variant(
    "full",
    **sage_avo_model_kwargs(workflow),
)
operator_model.set_norm_stats(normalization)
optimizer = torch.optim.AdamW(operator_model.parameters(), lr=float(training["learning_rate"]))
as_tensor = lambda name: torch.tensor(normalization[name], dtype=torch.float32).view(1, 3, 1, 1)
physics_normalization = PhysicsNormalization(
    x_mean=as_tensor("x_mean"), x_std=as_tensor("x_std"),
    y_mean=as_tensor("y_mean"), y_std=as_tensor("y_std"),
)
physics_settings = physics_settings_from_config(workflow)
base_weights = loss_weights_from_config(workflow, physics_weight=training["loss_weights"]["physics"])
weights = curriculum_from_config(workflow).weights_for_epoch(base_weights, 0, training["epochs"])
operator_metrics = train_step(
    operator_model, batch, optimizer, physics_normalization, weights,
    gradient_clip=float(training["gradient_clip"]),
    time_generator=torch.Generator().manual_seed(int(workflow["experiment"]["seed"]) + 17),
    physics=physics_settings,
)
display(pd.Series(operator_metrics.__dict__).to_frame("one-step value"))
assert all(np.isfinite(value) for value in operator_metrics.__dict__.values())

## 7. Weighted sampling, augmentation, and production training

Training uses replacement sampling weighted by reservoir-facies fraction, RGT-gradient complexity, and absolute AVO gradient. Registered augmentation applies horizontal geological flips, mild normalized AVO gain, and mild normalized noise. Validation uses no augmentation and the deterministic interior-time grid `[0.2, 0.5, 0.8]`.

Production training is disabled by default to prevent accidental GPU-scale execution. Setting `SAGE_AVO_RUN_PRODUCTION_TRAINING=1` activates the configured run after its required manifests pass validation. Every epoch logs raw loss components, current weighted terms, and the fixed-final-weight objective. Deterministic sampled metrics are separate. At configured intervals, fixed complete validation realizations are tiled and scored without test data. The run writes `best_fixed_objective.pt`, `best_sampling.pt`, `best_segmentation.pt`, `best_whole_realization.pt`, periodic checkpoints, and `last.pt`; every file records its criterion formula and resume state.

In [ ]:
run_production_training = os.getenv("SAGE_AVO_RUN_PRODUCTION_TRAINING", "0") == "1"
if run_production_training:
    run_directory = train_controlled_variant(
        repository=ROOT,
        config_path=workflow_path,
        config=workflow,
        dataset_directory=dataset_dir,
        experiment_directory=experiment_dir,
        variant="full",
    )
else:
    print("Production training is disabled (SAGE_AVO_RUN_PRODUCTION_TRAINING=0).")
    print("Set SAGE_AVO_RUN_PRODUCTION_TRAINING=1 to activate the configured full-model run.")

run_dir = experiment_dir / "runs" / (
    "full_2epoch_cuda_sanity" if validation_root_text else "full"
)
manifest_file = run_dir / "manifest.json"
display(pd.Series({
    "manifest": manifest_file.exists(),
    "best_fixed_objective_checkpoint": (run_dir / "best_fixed_objective.pt").exists(),
    "best_sampling_checkpoint": (run_dir / "best_sampling.pt").exists(),
    "best_segmentation_checkpoint": (run_dir / "best_segmentation.pt").exists(),
    "best_whole_realization_checkpoint": (run_dir / "best_whole_realization.pt").exists(),
    "resumable_last_checkpoint": (run_dir / "last.pt").exists(),
    "status": json.loads(manifest_file.read_text()).get("status") if manifest_file.exists() else "not_generated",
}).to_frame("value"))

## Stage outputs

| artifact | shape/type | scientific meaning | consumed by |
|---|---|---|---|
| `runs/full/best_fixed_objective.pt` | complete checkpoint state | Minimum fixed-final-weight patch-validation objective | Notebook 05 |
| `runs/full/best_sampling.pt` | model/optimizer/scheduler/RNG state | Checkpoint selected by sampled elastic RMSE and segmentation mIoU | Notebook 05 |
| `runs/full/best_segmentation.pt` | complete checkpoint state | Maximum deterministic sampled macro mIoU | Notebook 05 |
| `runs/full/best_whole_realization.pt` | complete checkpoint state | Preferred fixed whole-validation-section criterion | Notebook 05 |
| `runs/full/last.pt` | complete resumable state | Exact continuation point after the last completed epoch | training resume |
| `training_log.csv` | epoch-level objective terms and validation criteria | Optimization/QC history | Notebook 05 |
| `manifest.json` | seed, split IDs, normalization, prior, commit/config hash | Reproducibility and comparability record | Notebook 05 |

## Scientific checks

- A production-shape real batch passes the CNN, RGT graph, `TransformerConv`, dual decoders, differentiable exact-PP physics loss, and backward optimization.
- The straight-path target is asserted to equal `truth − low prior`.
- Physics guidance is executable with installed train-only normalization; zero guidance follows the identical unguided Heun path.
- Weighted sampling and training-only augmentation are configured explicitly and excluded from validation.
- Raw losses, current/fixed weighted contributions, sampled metrics, per-class metrics, and whole-validation metrics are logged separately.
- Criterion names/formulas and all checkpoint-selection minima/maxima are restored on resume; test data never selects a checkpoint.
- Operator validation is kept distinct from completed model training and scientific performance.

## Next stage

Notebook 05 consumes a criterion-selected full-model checkpoint for whole-realization synthetic inference and field deployment/QC. Controlled ablation and baseline comparisons require complete matched checkpoints.